### Load the Embedding Model

In [ ]:
from src.models import get_embedding_models

langchain_embedding_model, llamaindex_embedding_model = get_embedding_models()

### Load the documents

In [ ]:
from src.data_loader import load_text_document

doc = load_text_document("NVDA.txt")

### Chunk the documents with Semantic Splitter

In [ ]:
from src.chunking import get_semantic_splitter, split_documents

splitter = get_semantic_splitter(llamaindex_embedding_model)
nodes = split_documents(splitter, [doc])

print(f"Number of chunks: {len(nodes)}")

### Embed the chunks

In [ ]:
from src.chunking import extract_chunk_texts
from src.retrieval import embed_chunks

chunk_texts = extract_chunk_texts(nodes)
chunk_embeddings = embed_chunks(langchain_embedding_model, chunk_texts)

### Embed the query

In [ ]:
from src.retrieval import embed_query

query = "Why might Nvidia be overvalued?"

query_embedding = embed_query(langchain_embedding_model, query)

### Compute the similarity score

In [ ]:
from src.retrieval import compute_similarities

similarities = compute_similarities(query_embedding, chunk_embeddings)

### Defining the LLM

In [ ]:
from src.models import get_llm

llm = get_llm()

### Prompting the LLM

In [ ]:
from src.rag_pipeline import ask_rag

query = "Do you think I should invest in Nvidia now? Do you think it can grow in the future? I bought it earlier for 160 and sold it thinking there is an AI bubble."

result = ask_rag(
    query=query,
    chunk_texts=chunk_texts,
    chunk_embeddings=chunk_embeddings,
    embed_model=langchain_embedding_model,
    llm=llm,
    k=2
)

In [ ]:
for item in result["retrieved_chunks"]:
    print(f"\n=== Chunk {item['chunk_id']} | Score: {item['score']:.4f} ===\n")
    print(item["text"][:1000])

In [ ]:
print(result["prompt"])

In [ ]:
print(result["answer"])